# Bibliotecas

In [ ]:
# Importação das bibliotecas necessárias
import os # Para manipulação de arquivos e diretórios
import pandas as pd # Para manipulação de dados
import matplotlib.pyplot as plt # Para visualização de dados
import seaborn as sns # Para visualização de dados
from sklearn.tree import DecisionTreeRegressor # Para construção do modelo de regressão
from sklearn.ensemble import RandomForestRegressor # Para construção do modelo de regressão
from sklearn.metrics import mean_squared_error # Para avaliação do modelo
from sklearn.model_selection import train_test_split # Para divisão dos dados em treino e teste
from sklearn.preprocessing import OneHotEncoder # Para transformação de variáveis categóricas em numéricas
import kagglehub # Para acesso ao Kaggle e download de datasets
import numpy as np # Para manipulação de arrays e operações matemáticas
import math # Para operações matemáticas
import warnings # Para controle de mensagens de aviso
warnings.filterwarnings('ignore') # Ignorar mensagens de aviso


c:\Users\Annab\OneDrive\Documentos\kaggle\Housing-Prices\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Importação dos dados e análise de dados

In [ ]:
path = kagglehub.competition_download('home-data-for-ml-course')
print("Path to competition files:", path)

Path to competition files: C:\Users\Annab\.cache\kagglehub\competitions\home-data-for-ml-course


In [ ]:
pd.set_option('display.max_columns', None) # Configuração para exibir todas as colunas do DataFrame.
pd.set_option('display.max_rows', None) # Configuração para exibir todas as linhas do DataFrame.

In [ ]:
treino = pd.read_csv(os.path.join(path, 'train.csv'))
teste = pd.read_csv(os.path.join(path, 'test.csv'))

- Agora vamos primeiramente analisar os dados e ver valores nulos.

In [ ]:
treino.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [ ]:
treino.isnull().sum().sort_values(ascending=False).head(20)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageYrBlt       81
GarageCond        81
GarageType        81
GarageFinish      81
GarageQual        81
BsmtFinType2      38
BsmtExposure      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
Id                 0
dtype: int64

In [ ]:
teste.isnull().sum().sort_values(ascending=False).head(20)

PoolQC          1456
MiscFeature     1408
Alley           1352
Fence           1169
MasVnrType       894
FireplaceQu      730
LotFrontage      227
GarageYrBlt       78
GarageQual        78
GarageFinish      78
GarageCond        78
GarageType        76
BsmtCond          45
BsmtQual          44
BsmtExposure      44
BsmtFinType1      42
BsmtFinType2      42
MasVnrArea        15
MSZoning           4
BsmtHalfBath       2
dtype: int64

- Podemos ver que há muitos valores nulos, neste como temos colunas que possuem mais até de 70% dos valores nulos precisamos eliminá-las para que não interfiram na generalização do modelo.

In [ ]:
colunas_eliminar_treino = treino.columns[treino.isnull().sum() > 1460 * 0.7]
treino = treino.drop(columns=colunas_eliminar_treino)

In [ ]:
colunas_eliminar_teste = teste.columns[teste.isnull().sum() > 1460 * 0.7]
teste = teste.drop(columns=colunas_eliminar_teste)

- Para as colunas que tem valores nulos abaixo de 60% vamos substituir pela media ou mediana

In [ ]:
treino.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,TotRmsAbvGrd,Fireplaces,GarageYrBlt,GarageCars,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1379.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,46.549315,567.240411,1057.429452,1162.626712,346.992466,5.844521,1515.463699,0.425342,0.057534,1.565068,0.382877,2.866438,1.046575,6.517808,0.613014,1978.506164,1.767123,472.980137,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,161.319273,441.866955,438.705324,386.587738,436.528436,48.623081,525.480383,0.518911,0.238753,0.550916,0.502885,0.815778,0.220338,1.625393,0.644666,24.689725,0.747315,213.804841,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,0.000000,0.000000,0.000000,334.000000,0.000000,0.000000,334.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,1900.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,0.000000,223.000000,795.750000,882.000000,0.000000,0.000000,1129.500000,0.000000,0.000000,1.000000,0.000000,2.000000,1.000000,5.000000,0.000000,1961.000000,1.000000,334.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,0.000000,477.500000,991.500000,1087.000000,0.000000,0.000000,1464.000000,0.000000,0.000000,2.000000,0.000000,3.000000,1.000000,6.000000,1.000000,1980.000000,2.000000,480.000000,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,0.000000,808.000000,1298.250000,1391.250000,728.000000,0.000000,1776.750000,1.000000,0.000000,2.000000,1.000000,3.000000,1.000000,7.000000,1.000000,2002.000000,2.000000,576.000000,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,1474.000000,2336.000000,6110.000000,4692.000000,2065.000000,572.000000,5642.000000,3.000000,2.000000,3.000000,2.000000,8.000000,3.000000,14.000000,3.000000,2010.000000,4.000000,1418.000000,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


- Vamos descobrir se há outliers pelos quartis

- primeiro vamos ver quais são os tipos de cada uma

In [ ]:
treino.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 77 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   LotShape       1460 non-null   str    
 7   LandContour    1460 non-null   str    
 8   Utilities      1460 non-null   str    
 9   LotConfig      1460 non-null   str    
 10  LandSlope      1460 non-null   str    
 11  Neighborhood   1460 non-null   str    
 12  Condition1     1460 non-null   str    
 13  Condition2     1460 non-null   str    
 14  BldgType       1460 non-null   str    
 15  HouseStyle     1460 non-null   str    
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuilt      1460

In [ ]:
objeto_treino = treino.select_dtypes(include='object').columns
print(objeto_treino)
objeto_teste = teste.select_dtypes(include='object').columns
print(objeto_teste)

Index(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'SaleType', 'SaleCondition'],
      dtype='str')
Index(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heatin

In [ ]:
treino[objeto_treino].head(10)

,MSZoning,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Heating,HeatingQC,CentralAir,Electrical,KitchenQual,Functional,FireplaceQu,GarageType,GarageFinish,GarageQual,GarageCond,PavedDrive,SaleType,SaleCondition
0,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,BrkFace,Gd,TA,PConc,Gd,TA,No,GLQ,Unf,GasA,Ex,Y,SBrkr,Gd,Typ,NaN,Attchd,RFn,TA,TA,Y,WD,Normal
1,RL,Pave,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,Gable,CompShg,MetalSd,MetalSd,NaN,TA,TA,CBlock,Gd,TA,Gd,ALQ,Unf,GasA,Ex,Y,SBrkr,TA,Typ,TA,Attchd,RFn,TA,TA,Y,WD,Normal
2,RL,Pave,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,BrkFace,Gd,TA,PConc,Gd,TA,Mn,GLQ,Unf,GasA,Ex,Y,SBrkr,Gd,Typ,TA,Attchd,RFn,TA,TA,Y,WD,Normal
3,RL,Pave,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,Gable,CompShg,Wd Sdng,Wd Shng,NaN,TA,TA,BrkTil,TA,Gd,No,ALQ,Unf,GasA,Gd,Y,SBrkr,Gd,Typ,Gd,Detchd,Unf,TA,TA,Y,WD,Abnorml
4,RL,Pave,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,BrkFace,Gd,TA,PConc,Gd,TA,Av,GLQ,Unf,GasA,Ex,Y,SBrkr,Gd,Typ,TA,Attchd,RFn,TA,TA,Y,WD,Normal
5,RL,Pave,IR1,Lvl,AllPub,Inside,Gtl,Mitchel,Norm,Norm,1Fam,1.5Fin,Gable,CompShg,VinylSd,VinylSd,NaN,TA,TA,Wood,Gd,TA,No,GLQ,Unf,GasA,Ex,Y,SBrkr,TA,Typ,NaN,Attchd,Unf,TA,TA,Y,WD,Normal
6,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,Somerst,Norm,Norm,1Fam,1Story,Gable,CompShg,VinylSd,VinylSd,Stone,Gd,TA,PConc,Ex,TA,Av,GLQ,Unf,GasA,Ex,Y,SBrkr,Gd,Typ,Gd,Attchd,RFn,TA,TA,Y,WD,Normal
7,RL,Pave,IR1,Lvl,AllPub,Corner,Gtl,NWAmes,PosN,Norm,1Fam,2Story,Gable,CompShg,HdBoard,HdBoard,Stone,TA,TA,CBlock,Gd,TA,Mn,ALQ,BLQ,GasA,Ex,Y,SBrkr,TA,Typ,TA,Attchd,RFn,TA,TA,Y,WD,Normal
8,RM,Pave,Reg,Lvl,AllPub,Inside,Gtl,OldTown,Artery,Norm,1Fam,1.5Fin,Gable,CompShg,BrkFace,Wd Shng,NaN,TA,TA,BrkTil,TA,TA,No,Unf,Unf,GasA,Gd,Y,FuseF,TA,Min1,TA,Detchd,Unf,Fa,TA,Y,WD,Abnorml
9,RL,Pave,Reg,Lvl,AllPub,Corner,Gtl,BrkSide,Artery,Artery,2fmCon,1.5Unf,Gable,CompShg,MetalSd,MetalSd,NaN,TA,TA,BrkTil,TA,TA,No,GLQ,Unf,GasA,Ex,Y,SBrkr,TA,Typ,TA,Attchd,RFn,Gd,TA,Y,WD,Normal


In [ ]:
teste[objeto_teste].head(10)

,MSZoning,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Heating,HeatingQC,CentralAir,Electrical,KitchenQual,Functional,FireplaceQu,GarageType,GarageFinish,GarageQual,GarageCond,PavedDrive,SaleType,SaleCondition
0,RH,Pave,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,Gable,CompShg,VinylSd,VinylSd,NaN,TA,TA,CBlock,TA,TA,No,Rec,LwQ,GasA,TA,Y,SBrkr,TA,Typ,NaN,Attchd,Unf,TA,TA,Y,WD,Normal
1,RL,Pave,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,TA,TA,CBlock,TA,TA,No,ALQ,Unf,GasA,TA,Y,SBrkr,Gd,Typ,NaN,Attchd,Unf,TA,TA,Y,WD,Normal
2,RL,Pave,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,NaN,TA,TA,PConc,Gd,TA,No,GLQ,Unf,GasA,Gd,Y,SBrkr,TA,Typ,TA,Attchd,Fin,TA,TA,Y,WD,Normal
3,RL,Pave,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,BrkFace,TA,TA,PConc,TA,TA,No,GLQ,Unf,GasA,Ex,Y,SBrkr,Gd,Typ,Gd,Attchd,Fin,TA,TA,Y,WD,Normal
4,RL,Pave,IR1,HLS,AllPub,Inside,Gtl,StoneBr,Norm,Norm,TwnhsE,1Story,Gable,CompShg,HdBoard,HdBoard,NaN,Gd,TA,PConc,Gd,TA,No,ALQ,Unf,GasA,Ex,Y,SBrkr,Gd,Typ,NaN,Attchd,RFn,TA,TA,Y,WD,Normal
5,RL,Pave,IR1,Lvl,AllPub,Corner,Gtl,Gilbert,Norm,Norm,1Fam,2Story,Gable,CompShg,HdBoard,HdBoard,NaN,TA,TA,PConc,Gd,TA,No,Unf,Unf,GasA,Gd,Y,SBrkr,TA,Typ,TA,Attchd,Fin,TA,TA,Y,WD,Normal
6,RL,Pave,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,1Story,Gable,CompShg,HdBoard,HdBoard,NaN,TA,Gd,PConc,Gd,TA,No,ALQ,Unf,GasA,Ex,Y,SBrkr,TA,Typ,NaN,Attchd,Fin,TA,TA,Y,WD,Normal
7,RL,Pave,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,NaN,TA,TA,PConc,Gd,TA,No,Unf,Unf,GasA,Gd,Y,SBrkr,TA,Typ,Gd,Attchd,Fin,TA,TA,Y,WD,Normal
8,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,1Story,Gable,CompShg,HdBoard,HdBoard,NaN,TA,TA,PConc,Gd,TA,Gd,GLQ,Unf,GasA,Gd,Y,SBrkr,Gd,Typ,Po,Attchd,Unf,TA,TA,Y,WD,Normal
9,RL,Pave,Reg,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,Gable,CompShg,Plywood,Plywood,NaN,TA,TA,CBlock,TA,TA,No,ALQ,Rec,GasA,TA,Y,SBrkr,TA,Typ,NaN,Attchd,Fin,TA,TA,Y,WD,Normal


- Como possuimos strings de todos os tipos (ordinais e não ordinais) o jeito seriamos utilizar o tratamento OneHotEncoding - fazer esse tratamento no treino e teste

In [ ]:
for coluna in objeto_treino:

    print(treino.groupby([coluna]).size())
    print('='*50)

MSZoning
C (all)      10
FV           65
RH           16
RL         1151
RM          218
dtype: int64
Street
Grvl       6
Pave    1454
dtype: int64
LotShape
IR1    484
IR2     41
IR3     10
Reg    925
dtype: int64
LandContour
Bnk      63
HLS      50
Low      36
Lvl    1311
dtype: int64
Utilities
AllPub    1459
NoSeWa       1
dtype: int64
LotConfig
Corner      263
CulDSac      94
FR2          47
FR3           4
Inside     1052
dtype: int64
LandSlope
Gtl    1382
Mod      65
Sev      13
dtype: int64
Neighborhood
Blmngtn     17
Blueste      2
BrDale      16
BrkSide     58
ClearCr     28
CollgCr    150
Crawfor     51
Edwards    100
Gilbert     79
IDOTRR      37
MeadowV     17
Mitchel     49
NAmes      225
NPkVill      9
NWAmes      73
NoRidge     41
NridgHt     77
OldTown    113
SWISU       25
Sawyer      74
SawyerW     59
Somerst     86
StoneBr     25
Timber      38
Veenker     11
dtype: int64
Condition1
Artery      48
Feedr       81
Norm      1260
PosA         8
PosN        19
RRAe       

In [ ]:
for coluna in objeto_teste:

    print(teste.groupby([coluna]).size())
    print('='*50)

MSZoning
C (all)      15
FV           74
RH           10
RL         1114
RM          242
dtype: int64
Street
Grvl       6
Pave    1453
dtype: int64
LotShape
IR1    484
IR2     35
IR3      6
Reg    934
dtype: int64
LandContour
Bnk      54
HLS      70
Low      24
Lvl    1311
dtype: int64
Utilities
AllPub    1457
dtype: int64
LotConfig
Corner      248
CulDSac      82
FR2          38
FR3          10
Inside     1081
dtype: int64
LandSlope
Gtl    1396
Mod      60
Sev       3
dtype: int64
Neighborhood
Blmngtn     11
Blueste      8
BrDale      14
BrkSide     50
ClearCr     16
CollgCr    117
Crawfor     52
Edwards     94
Gilbert     86
IDOTRR      56
MeadowV     20
Mitchel     65
NAmes      218
NPkVill     14
NWAmes      58
NoRidge     30
NridgHt     89
OldTown    126
SWISU       23
Sawyer      77
SawyerW     66
Somerst     96
StoneBr     26
Timber      34
Veenker     13
dtype: int64
Condition1
Artery      44
Feedr       83
Norm      1251
PosA        12
PosN        20
RRAe        17
RRAn       

In [ ]:
#Utilizando o One-Hot Encoding para transformar as variáveis categóricas em numéricas
print(treino.shape)
print(teste.shape)

(1460, 77)
(1459, 76)


In [ ]:
# No teste não tem salesprice, pois será o que queremos prever.
print(treino.columns)
print(teste.columns)

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond',
       'PavedDrive', 'WoodD

In [ ]:
# One-Hot Encoding para o conjunto de treino
OHE = OneHotEncoder()

for coluna in objeto_treino:
    treino[coluna] = OHE.fit_transform(treino[[coluna]]).toarray() # transforma uma matriz esparsa em densa.

In [ ]:
treino[objeto_treino].head(10)

,MSZoning,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Heating,HeatingQC,CentralAir,Electrical,KitchenQual,Functional,FireplaceQu,GarageType,GarageFinish,GarageQual,GarageCond,PavedDrive,SaleType,SaleCondition
0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# One-Hot Encoding para o conjunto de treino

for coluna in objeto_teste:
    teste[coluna] = OHE.fit_transform(teste[[coluna]]).toarray() # transforma uma matriz esparsa em densa.

In [ ]:
teste[objeto_teste].head(10)

,MSZoning,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Heating,HeatingQC,CentralAir,Electrical,KitchenQual,Functional,FireplaceQu,GarageType,GarageFinish,GarageQual,GarageCond,PavedDrive,SaleType,SaleCondition
0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
7,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [ ]:

for coluna in treino.columns:
    if treino[coluna].isnull().sum() > 0 and coluna not in objeto_treino:
        print(coluna)

LotFrontage
MasVnrArea
GarageYrBlt


In [ ]:
treino.columns[treino.isnull().sum() > 0]

Index(['LotFrontage', 'MasVnrArea', 'GarageYrBlt'], dtype='str')

In [ ]:

for coluna in teste.columns:
    if teste[coluna].isnull().sum() > 0 and coluna not in objeto_teste:
        print(coluna)

LotFrontage
MasVnrArea
BsmtFinSF1
BsmtFinSF2
BsmtUnfSF
TotalBsmtSF
BsmtFullBath
BsmtHalfBath
GarageYrBlt
GarageCars
GarageArea


In [ ]:
teste.columns[teste.isnull().sum() > 0]

Index(['LotFrontage', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageYrBlt',
       'GarageCars', 'GarageArea'],
      dtype='str')

- Podemos ver que agora só sobrou valores nulos que são de colunas de variáveis numéricas

In [ ]:
treino.isnull().sum().sort_values(ascending=False).head(5)

LotFrontage     259
GarageYrBlt      81
MasVnrArea        8
Id                0
BedroomAbvGr      0
dtype: int64

In [ ]:
teste.isnull().sum().sort_values(ascending=False).head(15)

LotFrontage     227
GarageYrBlt      78
MasVnrArea       15
BsmtHalfBath      2
BsmtFullBath      2
TotalBsmtSF       1
BsmtUnfSF         1
BsmtFinSF2        1
BsmtFinSF1        1
GarageCars        1
GarageArea        1
BedroomAbvGr      0
KitchenQual       0
KitchenAbvGr      0
Id                0
dtype: int64

In [ ]:
#tratando os outliers
treino.describe()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
count,1460.000000,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1379.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,0.006849,70.049958,10516.828082,0.004110,0.331507,0.043151,0.999315,0.180137,0.946575,0.011644,0.032877,0.001370,0.835616,0.105479,6.099315,5.575342,1971.267808,1984.865753,0.008904,0.000685,0.013699,0.013699,0.010274,103.685262,0.035616,0.002055,0.100000,0.082877,0.030822,0.151370,0.150685,443.639726,0.013014,46.549315,567.240411,1057.429452,0.000685,0.507534,0.065068,0.064384,1162.626712,346.992466,5.844521,1515.463699,0.425342,0.057534,1.565068,0.382877,2.866438,1.046575,0.068493,6.517808,0.009589,0.613014,0.016438,0.004110,1978.506164,0.241096,1.767123,472.980137,0.002055,0.001370,0.061644,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,0.029452,0.069178,180921.195890
std,421.610009,42.300571,0.082505,24.284752,9981.264932,0.063996,0.470916,0.203266,0.026171,0.384433,0.224956,0.107313,0.178375,0.036999,0.370750,0.307275,1.382997,1.112799,30.202904,20.645407,0.093973,0.026171,0.116277,0.116277,0.100873,181.066207,0.185395,0.045299,0.300103,0.275790,0.172894,0.358532,0.357864,456.098091,0.113372,161.319273,441.866955,438.705324,0.026171,0.500115,0.246731,0.245519,386.587738,436.528436,48.623081,525.480383,0.518911,0.238753,0.550916,0.502885,0.815778,0.220338,0.252677,1.625393,0.097486,0.644666,0.127198,0.063996,24.689725,0.427895,0.747315,213.804841,0.045299,0.036999,0.240590,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,0.169128,0.253844,79442.502883
min,1.000000,20.000000,0.000000,21.000000,1300.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,334.000000,0.000000,0.000000,334.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,1900.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,0.000000,0.000000,34900.000000
25%,365.750000,20.000000,0.000000,59

In [ ]:
colunas_objetos = treino.select_dtypes(include='object').columns
print(colunas_objetos)
colunas_objetos = teste.select_dtypes(include='object').columns
print(colunas_objetos)

Index([], dtype='str')
Index([], dtype='str')


In [ ]:
# Agora vamos analisar os outliers presentes por meio dos quartis - treino.
colunas_numericas = treino.select_dtypes(include='number').columns
treino[colunas_numericas] = treino[colunas_numericas].astype(float)

for coluna in colunas_numericas:
    Q1 = treino[coluna].quantile(0.25)
    Q3 = treino[coluna].quantile(0.75)
    IQR = Q3 - Q1
    #Definindo os limites inferior e superior para identificar os outliers
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    mediana = treino[coluna].median()
    if treino[coluna].max() > limite_superior or treino[coluna].min() < limite_inferior:
        outlier = (treino[coluna] > limite_superior) | (treino[coluna] < limite_inferior)
        treino.loc[outlier, coluna] = mediana


In [ ]:
# Agora vamos analisar os outliers presentes por meio dos quartis - teste.
Q1 = teste.quantile(0.25)
Q3 = teste.quantile(0.75)
IQR = Q3 - Q1
# Definindo os limites inferior e superior para identificar os outliers
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

for coluna in teste.columns:
    mediana = teste[coluna].median()
    if teste[coluna].max() > limite_superior[coluna] or teste[coluna].min() < limite_inferior[coluna]:
        outlier = (teste[coluna] > limite_superior[coluna]) | (teste[coluna] < limite_inferior[coluna])
        teste.loc[outlier, coluna] = mediana

In [ ]:
# Tratar valores nulos - treino
for coluna in treino.columns:
    if treino[coluna].isnull().sum() > 0:
        media = treino[coluna].mean()
        treino[coluna] = treino[coluna].fillna(media) # Preenche os valores nulos com a média da coluna.

# Tratar valores nulos - teste
for coluna in teste.columns:
    if teste[coluna].isnull().sum() > 0:
        media = teste[coluna].mean()
        teste[coluna] = teste[coluna].fillna(media) # Preenche os valores nulos com a média da coluna correspondente.

In [ ]:
treino.isnull().sum().sort_values(ascending=False).head(5)

Id             0
HalfBath       0
FireplaceQu    0
Fireplaces     0
Functional     0
dtype: int64

In [ ]:
teste.isnull().sum().sort_values(ascending=False).head(5)

Id              0
FullBath        0
Fireplaces      0
Functional      0
TotRmsAbvGrd    0
dtype: int64

- Fizemos os tratamentos dos dados de outliers, nulos e do tipo string. Agora podemos iniciar treinando nossos modelos, como os preços de casa se relaciona à um problema de regressão e possuia outliers, o melhor nesses casos poderia ser o random forest e árvore de decisão (com base em meus estudos).

In [ ]:
#Inicializando as variáveis de treino e teste
X = treino.drop(columns=['SalePrice']) # Variáveis independentes (todas as colunas exceto 'SalePrice')
y = treino['SalePrice'] # Variável dependente (coluna 'SalePrice')

treino_X, teste_X, treino_y, teste_y = train_test_split(X, y, test_size=0.3, random_state=42) #test_size é para definir a proporção do conjunto de teste em relação ao conjunto total, e random_state é para garantir a reprodutibilidade dos resultados.

In [ ]:
def get_rmse_a(máximo, treino_X, treino_Y, teste_X, teste_Y):
    modelo_árvore = DecisionTreeRegressor(max_leaf_nodes=máximo, random_state=42)
    modelo_árvore.fit(treino_X, treino_y) # Treinando o modelo de árvore de decisão com os dados de treino.
    previsoes_árvore = modelo_árvore.predict(teste_X) # Fazendo previsões com o modelo de árvore de decisão usando os dados de teste.
    mse_árvore = mean_squared_error(teste_y, previsoes_árvore) # Calculando o erro quadrático médio entre as previsões e os valores reais.
    rmse_árvore = math.sqrt(mse_árvore) # Calculando a raiz quadrada do erro quadrático médio para obter o RMSE.
    return rmse_árvore

In [ ]:
#Descobrindo o melhor valor para max_leaf_nodes
for max in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]:
    best = math.inf
    rmse_árvore = get_rmse_a(max, treino_X, treino_y, teste_X, teste_y)
    print(f"RMSE da Decision Tree Regressor para {max} folhas:\t\t{rmse_árvore}")
    if best > rmse_árvore:
        best = rmse_árvore
        melhor_max_leaf_nodes_a = max

print(f"Melhor valor para max_leaf_nodes na Decision Tree Regressor: {melhor_max_leaf_nodes_a} com RMSE: {best}")

RMSE da Decision Tree Regressor para 5 folhas:		37048.88164922599
RMSE da Decision Tree Regressor para 10 folhas:		34586.3476475122
RMSE da Decision Tree Regressor para 15 folhas:		34330.647767589
RMSE da Decision Tree Regressor para 20 folhas:		33259.783006751626
RMSE da Decision Tree Regressor para 25 folhas:		33884.76214697573
RMSE da Decision Tree Regressor para 30 folhas:		35170.604826532326
RMSE da Decision Tree Regressor para 35 folhas:		35260.98424289616
RMSE da Decision Tree Regressor para 40 folhas:		35066.449626704074
RMSE da Decision Tree Regressor para 45 folhas:		35489.6344955545
RMSE da Decision Tree Regressor para 50 folhas:		35174.37264184742
RMSE da Decision Tree Regressor para 55 folhas:		35187.253351909734
RMSE da Decision Tree Regressor para 60 folhas:		36121.364195473834
RMSE da Decision Tree Regressor para 65 folhas:		36540.94468411937
RMSE da Decision Tree Regressor para 70 folhas:		37015.739771773086
RMSE da Decision Tree Regressor para 75 folhas:		37454.415287

In [ ]:
def get_rmse_f(máximo, treino_X, treino_Y, teste_X, teste_Y):
    modelo_floresta = RandomForestRegressor(max_leaf_nodes=máximo, random_state=42)
    modelo_floresta.fit(treino_X, treino_y) # Treinando o modelo de árvore de decisão com os dados de treino.
    previsoes_floresta = modelo_floresta.predict(teste_X) # Fazendo previsões com o modelo de árvore de decisão usando os dados de teste.
    mse_floresta = mean_squared_error(teste_y, previsoes_floresta) # Calculando o erro quadrático médio entre as previsões e os valores reais.
    rmse_floresta = math.sqrt(mse_floresta) # Calculando a raiz quadrada do erro quadrático médio para obter o RMSE.
    return rmse_floresta

In [ ]:
#Descobrindo o melhor valor para max_leaf_nodes
for max in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]:
    rmse_floresta = get_rmse_f(max, treino_X, treino_y, teste_X, teste_y)
    print(f"RMSE da Random Forest Regressor para {max} folhas:\t\t {rmse_floresta}")
    if best > rmse_floresta:
        best = rmse_floresta
        melhor_max_leaf_nodes_f = max

print(f"Melhor valor para max_leaf_nodes na Random Forest Regressor: {melhor_max_leaf_nodes_f} com RMSE: {best}")

RMSE da Random Forest Regressor para 5 folhas:		 34914.079902278965
RMSE da Random Forest Regressor para 10 folhas:		 32186.249479226073
RMSE da Random Forest Regressor para 15 folhas:		 31457.027011044105
RMSE da Random Forest Regressor para 20 folhas:		 30870.7623274655
RMSE da Random Forest Regressor para 25 folhas:		 30458.689626798543
RMSE da Random Forest Regressor para 30 folhas:		 30307.727241990186
RMSE da Random Forest Regressor para 35 folhas:		 30085.155094351645
RMSE da Random Forest Regressor para 40 folhas:		 29954.98022506714
RMSE da Random Forest Regressor para 45 folhas:		 29821.14322832307
RMSE da Random Forest Regressor para 50 folhas:		 29747.415746840325
RMSE da Random Forest Regressor para 55 folhas:		 29645.92306214732
RMSE da Random Forest Regressor para 60 folhas:		 29576.55558087841
RMSE da Random Forest Regressor para 65 folhas:		 29541.9112880158
RMSE da Random Forest Regressor para 70 folhas:		 29501.616608062937
RMSE da Random Forest Regressor para 75 fol

- O modelo escolhido foi a Random Forest Regressor pelo seu RMSE menor que a da Decision Tree Regressor

- Agora faremos o tratamento dos dados de teste

In [ ]:
teste_final = pd.read_csv(os.path.join(path, 'test.csv'))

In [ ]:
teste_final.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,CBlock,TA,TA,No,Rec,468.0,LwQ,144.0,270.0,882.0,GasA,TA,Y,SBrkr,896,0,0,896,0.0,0.0,1,0,2,1,TA,5,Typ,0,NaN,Attchd,1961.0,Unf,1.0,730.0,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.0,TA,TA,CBlock,TA,TA,No,ALQ,923.0,Unf,0.0,406.0,1329.0,GasA,TA,Y,SBrkr,1329,0,0,1329,0.0,0.0,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,1958.0,Unf,1.0,312.0,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,PConc,Gd,TA,No,GLQ,791.0,Unf,0.0,137.0,928.0,GasA,Gd,Y,SBrkr,928,701,0,1629,0.0,0.0,2,1,3,1,TA,6,Typ,1,TA,Attchd,1997.0,Fin,2.0,482.0,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,6,6,1998,1998,Gable,CompShg,VinylSd,VinylSd,BrkFace,20.0,TA,TA,PConc,TA,TA,No,GLQ,602.0,Unf,0.0,324.0,926.0,GasA,Ex,Y,SBrkr,926,678,0,1604,0.0,0.0,2,1,3,1,Gd,7,Typ,1,Gd,Attchd,1998.0,Fin,2.0,470.0,TA,TA,Y,360,36,0,0,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,Inside,Gtl,StoneBr,Norm,Norm,TwnhsE,1Story,8,5,1992,1992,Gable,CompShg,HdBoard,HdBoard,NaN,0.0,Gd,TA,PConc,Gd,TA,No,ALQ,263.0,Unf,0.0,1017.0,1280.0,GasA,Ex,Y,SBrkr,1280,0,0,1280,0.0,0.0,2,0,2,1,Gd,5,Typ,0,NaN,Attchd,1992.0,RFn,2.0,506.0,TA,TA,Y,0,82,0,0,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


- Vamos analisar os valores que são objetos (caracteres e strings)

In [ ]:
objeto_teste_final = teste_final.select_dtypes(include='object').columns
print(objeto_teste_final)

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='str')


In [ ]:
#usaremos agora o one-hot encoding para o teste_final
for coluna in objeto_teste_final:
    teste_final[coluna] = OHE.fit_transform(teste_final[[coluna]]).toarray() # transforma uma matriz esparsa em densa.

In [ ]:
#Vamos ver o resultado do teste_final após o one-hot encoding
teste_final.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,0.0,0.0,11622,1.0,0.0,0.0,0.0,80.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,5,6,1961,1961,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,896,0,0,896,0.0,0.0,1,0,2,1,0.0,5,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,140,0,0,0,120,0,0.0,0.0,0.0,0,6,2010,0.0,0.0
1,1462,20,0.0,0.0,14267,0.0,1.0,0.0,0.0,81.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,6,6,1958,1958,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1329,0,0,1329,0.0,0.0,1,1,3,1,0.0,6,0.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,393,36,0,0,0,0,0.0,0.0,0.0,12500,6,2010,0.0,0.0
2,1463,60,0.0,0.0,13830,0.0,1.0,0.0,0.0,74.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,5,5,1997,1998,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,928,701,0,1629,0.0,0.0,2,1,3,1,1.0,6,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,212,34,0,0,0,0,0.0,0.0,0.0,0,3,2010,0.0,0.0
3,1464,60,0.0,0.0,9978,0.0,1.0,0.0,0.0,78.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,6,6,1998,1998,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,926,678,0,1604,0.0,0.0,2,1,3,1,1.0,7,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,360,36,0,0,0,0,0.0,0.0,0.0,0,6,2010,0.0,0.0
4,1465,120,0.0,0.0,5005,0.0,1.0,0.0,0.0,43.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,8,5,1992,1992,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1280,0,0,1280,0.0,0.0,2,0,2,1,0.0,5,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,82,0,0,144,0,0.0,0.0,0.0,0,1,2010,1.0,0.0


- Agora vamos ver a quantidade valores nulos

In [ ]:
teste_final.shape

(1459, 80)

In [ ]:
teste_final.isnull().sum().sort_values(ascending=False)

LotFrontage      227
GarageYrBlt       78
MasVnrArea        15
BsmtFullBath       2
BsmtHalfBath       2
GarageCars         1
TotalBsmtSF        1
BsmtUnfSF          1
BsmtFinSF2         1
BsmtFinSF1         1
GarageArea         1
KitchenQual        0
Fireplaces         0
Functional         0
TotRmsAbvGrd       0
Id                 0
KitchenAbvGr       0
BedroomAbvGr       0
HalfBath           0
FireplaceQu        0
GrLivArea          0
LowQualFinSF       0
FullBath           0
GarageFinish       0
GarageType         0
PoolArea           0
SaleType           0
YrSold             0
MoSold             0
MiscVal            0
MiscFeature        0
Fence              0
PoolQC             0
ScreenPorch        0
1stFlrSF           0
3SsnPorch          0
EnclosedPorch      0
OpenPorchSF        0
WoodDeckSF         0
PavedDrive         0
GarageCond         0
GarageQual         0
2ndFlrSF           0
HeatingQC          0
Electrical         0
CentralAir         0
OverallQual        0
HouseStyle   

- Tratamento de outliers

In [ ]:
# Agora vamos analisar os outliers presentes por meio dos quartis - teste.
Q1 = teste_final.quantile(0.25)
Q3 = teste_final.quantile(0.75)
IQR = Q3 - Q1
# Definindo os limites inferior e superior para identificar os outliers
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

for coluna in teste_final.columns:
    mediana = teste_final[coluna].median()
    if teste_final[coluna].max() > limite_superior[coluna] or teste_final[coluna].min() < limite_inferior[coluna]:
        outlier = (teste_final[coluna] > limite_superior[coluna]) | (teste_final[coluna] < limite_inferior[coluna])
        teste_final.loc[outlier, coluna] = mediana

In [ ]:
# Vamos agora subsituir valores nulos por média no teste_final
for coluna in teste_final.columns:
    if teste_final[coluna].isnull().sum() > 0:
        media = teste_final[coluna].mean()
        teste_final[coluna] = teste_final[coluna].fillna(media) # Preenche os valores nulos com a média da coluna correspondente.

In [ ]:
teste_final.isnull().sum().sort_values(ascending=False)

Id               0
MSSubClass       0
GarageType       0
FireplaceQu      0
Fireplaces       0
Functional       0
TotRmsAbvGrd     0
KitchenQual      0
KitchenAbvGr     0
BedroomAbvGr     0
HalfBath         0
FullBath         0
BsmtHalfBath     0
BsmtFullBath     0
GrLivArea        0
LowQualFinSF     0
2ndFlrSF         0
1stFlrSF         0
Electrical       0
GarageYrBlt      0
GarageFinish     0
GarageCars       0
PoolArea         0
SaleType         0
YrSold           0
MoSold           0
MiscVal          0
MiscFeature      0
Fence            0
PoolQC           0
ScreenPorch      0
GarageArea       0
3SsnPorch        0
EnclosedPorch    0
OpenPorchSF      0
WoodDeckSF       0
PavedDrive       0
GarageCond       0
GarageQual       0
CentralAir       0
HeatingQC        0
Heating          0
LotConfig        0
OverallQual      0
HouseStyle       0
BldgType         0
Condition2       0
Condition1       0
Neighborhood     0
LandSlope        0
Utilities        0
YearBuilt        0
LandContour 

- Utilização do modelo Random Forest

In [ ]:
modelo_final = RandomForestRegressor(max_leaf_nodes=melhor_max_leaf_nodes_f, random_state=42)
modelo_final.fit(treino_X, treino_y)
# 1. Armazene os IDs em uma variável separada ANTES de qualquer drop
ids_para_submissao = teste_final['Id']

# 2. Garanta que o teste_final tenha EXATAMENTE as mesmas colunas do treino_X
# Em vez de apenas dar drop, selecione as colunas que você usou no treino
X_teste_alinhado = teste_final[treino_X.columns]

# 3. Agora faça a predição usando os dados alinhados (sem a coluna Id)
predicoes_finais = modelo_final.predict(X_teste_alinhado)

# 4. Monte o DataFrame de submissão usando a variável de IDs que guardamos
submissao = pd.DataFrame({
    'Id': ids_para_submissao,
    'SalePrice': predicoes_finais
})

# 5. Salve o arquivo
submissao.to_csv('sample_submission.csv', index=False)
print('Submissão salva com sucesso!')

Submissão salva com sucesso!
